# Math Photo Solver — Обучение в Google Colab

Этот ноутбук выполняет полный цикл:
1. Клонирует репозиторий
2. Устанавливает зависимости
3. Генерирует датасет символов (80% train / 20% val)
4. Обучает классификатор ResNet-18
5. Оценивает точность модели
6. Сохраняет веса на Google Drive

> **Перед запуском**: `Среда выполнения → Сменить тип среды выполнения → GPU (T4) → Сохранить`

## Шаг 1. Подключение Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_SAVE_DIR = '/content/drive/MyDrive/math_solver_models'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print('Google Drive подключён. Модель будет сохранена в:', DRIVE_SAVE_DIR)

## Шаг 2. Клонирование репозитория

In [ ]:
# Замените URL на адрес своего репозитория
REPO_URL = 'https://github.com/sergey2321/sergey2321.git'

!git clone {REPO_URL} /content/math-solver
%cd /content/math-solver
!git checkout claude/ale-yP87K
print('Репозиторий склонирован.')

## Шаг 3. Установка зависимостей

In [ ]:
!pip install -q -r requirements.txt
print('Зависимости установлены.')

## Шаг 4. Проверка GPU

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Устройство: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('ВНИМАНИЕ: GPU не найден. Обучение будет медленным.')

## Шаг 5. Генерация датасета символов (80/20)

In [ ]:
# Количество изображений — увеличь для лучшей точности (рекомендуется 20000+)
DATASET_COUNT = 10000
DATASET_DIR   = 'dataset/symbols'

!python -m dataset.generator.generate_handwritten \
    --count {DATASET_COUNT} \
    --out {DATASET_DIR} \
    --seed 42

import os
train_count = len(os.listdir(f'{DATASET_DIR}/train'))
val_count   = len(os.listdir(f'{DATASET_DIR}/val'))
print(f'Train: {train_count}  |  Val: {val_count}')

## Шаг 6. Обучение модели

In [ ]:
EPOCHS     = 25
BATCH_SIZE = 128
LR         = 1e-3
MODEL_OUT  = 'backend/models/symbol_clf.pth'

!python -m training.train_ocr \
    --data {DATASET_DIR} \
    --epochs {EPOCHS} \
    --batch {BATCH_SIZE} \
    --lr {LR} \
    --out {MODEL_OUT}

## Шаг 7. Оценка точности модели

In [ ]:
!python -m training.evaluate \
    --data {DATASET_DIR} \
    --model {MODEL_OUT}

## Шаг 8. Сохранение модели на Google Drive

In [ ]:
import shutil
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d_%H%M')
dest = f'{DRIVE_SAVE_DIR}/symbol_clf_{timestamp}.pth'
shutil.copy(MODEL_OUT, dest)
print(f'Модель сохранена на Drive: {dest}')

# Также сохраняем под стандартным именем
shutil.copy(MODEL_OUT, f'{DRIVE_SAVE_DIR}/symbol_clf_latest.pth')
print(f'Также сохранена как: {DRIVE_SAVE_DIR}/symbol_clf_latest.pth')

## Шаг 9. Скачивание модели на локальный компьютер

После завершения обучения скачай веса одним из способов:

**Способ 1** — Скачать прямо из Colab (раскомментируй ячейку ниже).

**Способ 2** — Взять с Google Drive:  
`Мой диск → math_solver_models → symbol_clf_latest.pth`

Положи скачанный файл в `backend/models/symbol_clf.pth` в локальном репозитории — и рукописные задачи начнут распознаваться через твою нейросеть.

In [ ]:
# Раскомментируй для прямого скачивания
# from google.colab import files
# files.download('backend/models/symbol_clf.pth')

## Бонус: Визуализация датасета

Покажем несколько примеров сгенерированных символов.

In [ ]:
import json
import matplotlib.pyplot as plt
from PIL import Image
import random

with open(f'{DATASET_DIR}/metadata.json') as f:
    meta = json.load(f)

samples = random.sample(meta['train'], min(20, len(meta['train'])))

fig, axes = plt.subplots(2, 10, figsize=(20, 5))
for ax, rec in zip(axes.flat, samples):
    img = Image.open(f"{DATASET_DIR}/{rec['file']}")
    ax.imshow(img, cmap='gray')
    ax.set_title(rec['symbol'], fontsize=10)
    ax.axis('off')

plt.suptitle('Примеры символов из датасета', fontsize=14)
plt.tight_layout()
plt.show()